# Thematic Analysis Framework
## Phase 3: Semantic Embeddings

Input: `phase3_light_dataset.csv` (529,529 rows, `text_clean_light` column)

Turns each review into a dense vector using a pretrained sentence-transformer model, so that
reviews with similar *meaning* end up close together in vector space — even if they share
almost no words (e.g. "the lecturer explained well" vs "the instructor made difficult topics
easy"). These embeddings feed directly into BERTopic in Phase 4.

**Output:** `embeddings.npy` (the vectors, one row per review, in the same order as `review_id.npy`)
and `review_id.npy` (so you can always join an embedding back to its original review).

> **Colab tip:** Runtime → Change runtime type → **GPU** (T4 is fine). Embedding ~530K reviews on
> CPU is possible but slow; a GPU runtime will cut this from hours to minutes. This notebook
> checks which device it's actually running on before committing to the full run.

> **Reliability note:** free Colab sessions can disconnect mid-run on a job this size. This
> notebook encodes in **chunks** and saves each chunk to Drive as it finishes, so an interrupted
> run can be resumed rather than restarted from scratch.


## Setup

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

INPUT_PATH  = "/content/drive/MyDrive/ThesisDocuments/Preprocessing/phase3_light_dataset.csv"
OUTPUT_DIR  = "/content/drive/MyDrive/ThesisDocuments/Preprocessing/embeddings"

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1. Imports & Config

**Model choice:** `all-MiniLM-L6-v2` is the default — it's the model BERTopic's own
documentation and most published examples pair it with, it's fast, and its quality is more
than sufficient for topic clustering. `all-mpnet-base-v2` is a slower, higher-quality
alternative (768-dim vs 384-dim vectors) if you want to compare embedding quality later —
swap `EMBEDDING_MODEL` below if you want to try it, but expect roughly 2–3x the runtime.

In [ ]:
import numpy as np
import pandas as pd
import time
import glob
from sentence_transformers import SentenceTransformer
import torch

EMBEDDING_MODEL = "all-MiniLM-L6-v2"   # 384-dim, fast, BERTopic's standard pairing
CHUNK_SIZE      = 50_000               # rows per checkpoint -- tune down if you hit memory issues
ENCODE_BATCH    = 128                  # sentence-transformers internal batch size

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cpu":
    print("WARNING: no GPU detected. Go to Runtime -> Change runtime type -> GPU before the full run.")

model = SentenceTransformer(EMBEDDING_MODEL, device=device)

## 2. Load the Light-Cleaned Dataset

In [ ]:
df = pd.read_csv(INPUT_PATH)
df = df.sort_values("review_id").reset_index(drop=True)  # fix a stable, deterministic row order

print(f"Rows: {len(df):,}")
print(f"Any missing text_clean_light? {df['text_clean_light'].isna().sum()}")

texts = df["text_clean_light"].astype(str).tolist()
ids = df["review_id"].to_numpy()

## 3. Sanity Check

Before running on the full corpus, confirm the embeddings actually capture meaning: two
reviews saying the same thing in different words should have high cosine similarity, and
two unrelated reviews should not.

In [ ]:
from sentence_transformers.util import cos_sim

examples = [
    "The lecturer explained everything really clearly.",
    "The instructor made difficult topics easy to understand.",
    "The pricing of this course was way too expensive for what it offered.",
]
example_emb = model.encode(examples)

sim_related = cos_sim(example_emb[0], example_emb[1]).item()
sim_unrelated = cos_sim(example_emb[0], example_emb[2]).item()

print(f"Similarity (both about teaching clarity): {sim_related:.3f}  <- should be high")
print(f"Similarity (teaching vs pricing):          {sim_unrelated:.3f}  <- should be lower")

assert sim_related > sim_unrelated, "Sanity check failed -- embeddings may not be capturing meaning as expected."
print("\nSanity check passed.")

## 4. Time Estimate

Times a small sample before committing to the full run, so you know roughly how long to expect
(and whether you're actually on a GPU).

In [ ]:
sample_texts = texts[:2000]
t0 = time.time()
_ = model.encode(sample_texts, batch_size=ENCODE_BATCH, show_progress_bar=False)
t1 = time.time()

rate = (t1 - t0) / len(sample_texts)
print(f"{len(sample_texts):,} rows took {t1 - t0:.1f}s ({rate*1000:.2f} ms/row)")
print(f"Estimated full run ({len(texts):,} rows): {rate * len(texts) / 60:.1f} minutes")

## 5. Encode in Chunks (resumable)

Encodes `CHUNK_SIZE` rows at a time and saves each chunk to `OUTPUT_DIR` immediately. If this
cell is interrupted (e.g. Colab disconnects) and you re-run it, it **skips chunks that already
have a saved file** and picks up where it left off — you won't lose completed work.

In [ ]:
n = len(texts)
n_chunks = (n + CHUNK_SIZE - 1) // CHUNK_SIZE
print(f"Total chunks: {n_chunks} (chunk size {CHUNK_SIZE:,})")

t_start = time.time()
for chunk_idx in range(n_chunks):
    chunk_path = os.path.join(OUTPUT_DIR, f"chunk_{chunk_idx:04d}.npy")
    if os.path.exists(chunk_path):
        print(f"Chunk {chunk_idx+1}/{n_chunks} already done, skipping.")
        continue

    start = chunk_idx * CHUNK_SIZE
    end = min(start + CHUNK_SIZE, n)
    chunk_texts = texts[start:end]

    t0 = time.time()
    chunk_emb = model.encode(
        chunk_texts,
        batch_size=ENCODE_BATCH,
        show_progress_bar=False,
        convert_to_numpy=True,
    )
    np.save(chunk_path, chunk_emb)
    print(f"Chunk {chunk_idx+1}/{n_chunks} ({start:,}-{end:,}) done in {time.time()-t0:.1f}s")

print(f"\nAll chunks complete. Total time: {(time.time()-t_start)/60:.1f} minutes")

## 6. Stitch Chunks Into Final Arrays

Combines all chunk files into one `embeddings.npy` array, aligned row-for-row with `review_id.npy`.

In [ ]:
chunk_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "chunk_*.npy")))
assert len(chunk_files) == n_chunks, f"Expected {n_chunks} chunk files, found {len(chunk_files)} -- re-run Section 5 first."

embeddings = np.concatenate([np.load(f) for f in chunk_files], axis=0)
print(f"Final embeddings shape: {embeddings.shape}")
assert embeddings.shape[0] == len(ids), "Row count mismatch between embeddings and review_id -- do not proceed to Phase 4 until this matches."

np.save(os.path.join(OUTPUT_DIR, "embeddings.npy"), embeddings)
np.save(os.path.join(OUTPUT_DIR, "review_id.npy"), ids)
print(f"Saved embeddings.npy {embeddings.shape} and review_id.npy {ids.shape} to {OUTPUT_DIR}")

## 7. (Optional) Clean Up Chunk Files

Once `embeddings.npy` is saved and verified, the individual chunk files are redundant and can
be removed to save Drive space. Only run this after you've confirmed Section 6 completed
successfully.

In [ ]:
# Uncomment to delete the intermediate chunk files
# for f in chunk_files:
#     os.remove(f)
# print(f"Removed {len(chunk_files)} chunk files.")